# Retrieval Fundamentals

This notebook teaches the **retriever** in modern NLP systems. We will build intuition for lexical search, semantic search, chunking, approximate nearest neighbor search, and the metrics used to judge whether a retriever is actually useful.

By the end, you will understand:

1. Why retrieval exists as a separate subsystem
2. How **chunk size** changes retrieval quality
3. How **BM25** ranks text from term statistics
4. How a tiny **dual encoder** learns semantic retrieval locally
5. Why ANN indexes trade a little recall for much lower search cost
6. How to interpret **Recall@k**, **Precision@k**, **MRR**, and **nDCG**

This notebook stops at retrieval. The next notebook, **RAG**, uses these retrieval ideas inside a full retriever-plus-generator pipeline.


## 1. Setup

We keep all experiment settings in one place so the notebook is easy to rerun and modify.


In [ ]:
import math
import random
import re
import time
from collections import Counter
from dataclasses import dataclass
import importlib.util
from pathlib import Path

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import CSVLogger
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

repo_root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
utils_path = repo_root / 'src/aiml_notebooks/utils.py'
spec = importlib.util.spec_from_file_location('aiml_notebooks_utils', utils_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)
get_device = utils.get_device
set_seed = utils.set_seed

CONFIG = {
    'seed': 42,
    'max_length': 24,
    'batch_size': 8,
    'embedding_dim': 48,
    'projection_dim': 32,
    'learning_rate': 3e-3,
    'max_epochs': 20,
    'early_stop_patience': 4,
    'temperature': 0.15,
    'top_k': 3,
    'ann_num_clusters': 6,
    'ann_clusters_to_search': 2,
    'latency_repeats': 300,
}

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 140)
print('Configuration loaded.')


### Random Seed and Device

The dense retriever is intentionally tiny, so CPU execution is fast enough and avoids transformer-specific MPS issues.


In [ ]:
set_seed(CONFIG['seed'])
L.seed_everything(CONFIG['seed'], workers=True)
device = get_device(prefer_cpu=True)
print(f'Chosen device: {device}')


## 2. Build a Retrieval Playground

A good retrieval notebook needs **controlled failures**. Some queries should match exact words, while others should require semantic reasoning or better chunking.


In [ ]:
@dataclass
class Document:
    doc_id: str
    topic: str
    title: str
    text: str


DOCUMENTS = [
    Document('doc_billing_refund', 'billing', 'Refund policy',
             'Customers can request a refund within thirty days of purchase. '
             'Reimbursements are sent to the original payment method. '
             'Processing usually takes five business days.'),
    Document('doc_billing_double_charge', 'billing', 'Duplicate charge troubleshooting',
             'If a customer sees the same charge twice, support should verify pending authorizations first. '
             'A duplicate transaction may disappear automatically within forty eight hours. '
             'Escalate confirmed duplicate charges to the billing team.'),
    Document('doc_shipping_delay', 'shipping', 'Delayed delivery updates',
             'A delayed package may still be moving between regional hubs. '
             'Customers should review the latest tracking scan before opening a lost parcel claim. '
             'Expedited replacements are available for urgent shipments.'),
    Document('doc_shipping_damage', 'shipping', 'Damaged parcel procedure',
             'If an item arrives damaged, photograph the parcel and packaging immediately. '
             'Support can arrange a replacement shipment after reviewing the evidence. '
             'Severely damaged orders may qualify for a refund instead of a resend.'),
    Document('doc_account_password', 'account', 'Password reset flow',
             'Users who cannot sign in should begin with the password reset link. '
             'Reset instructions are sent to the verified email address. '
             'Support agents should confirm account ownership before manual changes.'),
    Document('doc_account_2fa', 'account', 'Two factor authentication help',
             'Two factor authentication adds a second identity check during sign in. '
             'People who lose their authenticator device can use backup codes to recover access. '
             'If backup codes are unavailable, identity verification is required.'),
    Document('doc_membership_cancel', 'membership', 'Membership cancellation',
             'Members can cancel the subscription before the next renewal date. '
             'Cancellation stops future billing but does not refund the current cycle automatically. '
             'The confirmation email serves as proof that the plan was terminated.'),
    Document('doc_membership_pause', 'membership', 'Subscription pause option',
             'Some annual plans can be paused for up to two months. '
             'A pause temporarily suspends shipments without ending the membership. '
             'Billing resumes automatically after the pause window closes.'),
    Document('doc_security_login_alert', 'security', 'Unexpected login alert',
             'Security alerts appear when we detect sign in attempts from unfamiliar devices. '
             'Customers should change credentials and review recent sessions if the alert was not expected. '
             'High risk activity may lock the account until verification finishes.'),
    Document('doc_returns_exchange', 'returns', 'Exchange and replacement policy',
             'Customers can exchange products that arrive defective or incorrect. '
             'Replacement items ship after the returned package is scanned. '
             'Support may waive the return label fee for fulfillment mistakes.'),
    Document('doc_tracking_address', 'shipping', 'Address correction process',
             'Address changes are possible only before the parcel reaches the final carrier. '
             'Once the last mile handoff happens, the package may need to be rerouted by the carrier directly. '
             'Support should verify the corrected address carefully.'),
    Document('doc_billing_invoice', 'billing', 'Invoice and receipt requests',
             'Receipts and invoices are available in the billing portal. '
             'Enterprise administrators can download monthly statements in PDF format. '
             'Support can resend a missing receipt by email.'),
]

EVAL_QUERIES = [
    {'query_id': 'q_refund_window', 'text': 'How long do customers have to ask for a reimbursement?', 'relevant_docs': ['doc_billing_refund'], 'intent': 'semantic synonym'},
    {'query_id': 'q_double_charge', 'text': 'A customer says they were charged twice for the same order.', 'relevant_docs': ['doc_billing_double_charge'], 'intent': 'lexical'},
    {'query_id': 'q_lost_package', 'text': 'What should support do when a parcel is running late?', 'relevant_docs': ['doc_shipping_delay'], 'intent': 'package vs parcel'},
    {'query_id': 'q_damaged_item', 'text': 'What is the process for an order that arrived broken?', 'relevant_docs': ['doc_shipping_damage', 'doc_returns_exchange'], 'intent': 'multi relevant'},
    {'query_id': 'q_password_help', 'text': 'How can someone recover access when they forgot their credentials?', 'relevant_docs': ['doc_account_password'], 'intent': 'credentials vs password'},
    {'query_id': 'q_2fa_backup', 'text': 'What should a user do after losing the phone used for two step verification?', 'relevant_docs': ['doc_account_2fa'], 'intent': '2FA paraphrase'},
    {'query_id': 'q_cancel_plan', 'text': 'How does a customer terminate their membership before renewal?', 'relevant_docs': ['doc_membership_cancel'], 'intent': 'terminate vs cancel'},
    {'query_id': 'q_pause_plan', 'text': 'Can a subscriber temporarily stop shipments without ending the plan?', 'relevant_docs': ['doc_membership_pause'], 'intent': 'pause semantics'},
    {'query_id': 'q_security_alert', 'text': 'What should someone do after getting an unfamiliar sign in warning?', 'relevant_docs': ['doc_security_login_alert'], 'intent': 'warning vs alert'},
    {'query_id': 'q_exchange_policy', 'text': 'When can support replace a defective item instead of refunding it?', 'relevant_docs': ['doc_returns_exchange', 'doc_shipping_damage'], 'intent': 'replacement policy'},
    {'query_id': 'q_address_fix', 'text': 'Can support correct the delivery address after the shipment is already moving?', 'relevant_docs': ['doc_tracking_address'], 'intent': 'address correction'},
    {'query_id': 'q_receipt_copy', 'text': 'Where can an admin download a monthly invoice or receipt?', 'relevant_docs': ['doc_billing_invoice'], 'intent': 'receipt lookup'},
]

TRAIN_PAIRS = [
    {'query_text': 'How do I get a refund on a recent purchase?', 'doc_id': 'doc_billing_refund'},
    {'query_text': 'Where do reimbursements get sent?', 'doc_id': 'doc_billing_refund'},
    {'query_text': 'The card statement shows the same payment twice.', 'doc_id': 'doc_billing_double_charge'},
    {'query_text': 'What should support do with a duplicate transaction?', 'doc_id': 'doc_billing_double_charge'},
    {'query_text': 'What happens when a package is delayed in transit?', 'doc_id': 'doc_shipping_delay'},
    {'query_text': 'How should support handle a late parcel?', 'doc_id': 'doc_shipping_delay'},
    {'query_text': 'What should we do when an item arrives damaged?', 'doc_id': 'doc_shipping_damage'},
    {'query_text': 'How is a broken shipment replaced?', 'doc_id': 'doc_shipping_damage'},
    {'query_text': 'How can users reset a forgotten password?', 'doc_id': 'doc_account_password'},
    {'query_text': 'How do people recover their credentials?', 'doc_id': 'doc_account_password'},
    {'query_text': 'What if someone loses their two factor device?', 'doc_id': 'doc_account_2fa'},
    {'query_text': 'How can backup codes restore access?', 'doc_id': 'doc_account_2fa'},
    {'query_text': 'How do members cancel before renewal?', 'doc_id': 'doc_membership_cancel'},
    {'query_text': 'How can a customer terminate the subscription?', 'doc_id': 'doc_membership_cancel'},
    {'query_text': 'Can a plan be paused without cancellation?', 'doc_id': 'doc_membership_pause'},
    {'query_text': 'How do temporary shipment pauses work?', 'doc_id': 'doc_membership_pause'},
    {'query_text': 'What should a user do after an unfamiliar login alert?', 'doc_id': 'doc_security_login_alert'},
    {'query_text': 'How should people respond to suspicious sign in warnings?', 'doc_id': 'doc_security_login_alert'},
    {'query_text': 'When can support exchange a defective product?', 'doc_id': 'doc_returns_exchange'},
    {'query_text': 'How are replacement items handled after returns?', 'doc_id': 'doc_returns_exchange'},
]

docs_df = pd.DataFrame([doc.__dict__ for doc in DOCUMENTS])
queries_df = pd.DataFrame(EVAL_QUERIES)
train_pairs_df = pd.DataFrame(TRAIN_PAIRS)
print(f'Documents: {len(docs_df)} | Evaluation queries: {len(queries_df)} | Training pairs: {len(train_pairs_df)}')
display(queries_df[['query_id', 'text', 'relevant_docs', 'intent']])


### Inspect the Corpus Shape

Even simple retrieval corpora have topic skew and mixed intent types. It helps to understand the playground before measuring retrieval quality.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
docs_df['topic'].value_counts().sort_values().plot(kind='barh', ax=ax, color='#4C78A8')
ax.set_title('Document topics in the synthetic corpus')
ax.set_xlabel('Number of documents')
plt.tight_layout()
plt.show()


### Why Retrieval Exists

Retrieval systems are balancing four pressures at the same time:

- **lexical precision**: exact terms matter for some queries
- **semantic flexibility**: users paraphrase, use synonyms, and omit product jargon
- **latency**: scanning everything is too slow at scale
- **evaluation**: we need metrics that reflect whether relevant evidence is actually found


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pressures = pd.DataFrame({
    'dimension': ['Lexical match', 'Semantic match', 'Latency', 'Evaluation'],
    'importance': [9, 9, 8, 10],
})
ax.barh(pressures['dimension'], pressures['importance'], color=['#4C78A8', '#F58518', '#54A24B', '#E45756'])
ax.set_xlim(0, 10)
ax.set_title('Why retrieval is its own subsystem')
ax.set_xlabel('Importance in production systems (illustrative)')
plt.tight_layout()
plt.show()


## 3. Chunking Comes Before Retrieval

Retrievers search over **units**. In long-document systems, those units are usually chunks rather than full documents. Chunk size changes both recall and precision.


In [ ]:
def sentence_split(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def build_chunks(documents, sentences_per_chunk):
    rows = []
    for doc in documents:
        sentences = sentence_split(doc.text)
        for start in range(0, len(sentences), sentences_per_chunk):
            piece = sentences[start:start + sentences_per_chunk]
            rows.append({
                'chunk_id': f"{doc.doc_id}_chunk_{start // sentences_per_chunk}",
                'doc_id': doc.doc_id,
                'topic': doc.topic,
                'text': ' '.join(piece),
            })
    return pd.DataFrame(rows)

coarse_chunks = build_chunks(DOCUMENTS, sentences_per_chunk=3)
fine_chunks = build_chunks(DOCUMENTS, sentences_per_chunk=1)
chunk_summary = pd.DataFrame({
    'strategy': ['Coarse (3 sentences)', 'Fine (1 sentence)'],
    'num_chunks': [len(coarse_chunks), len(fine_chunks)],
    'avg_tokens': [coarse_chunks['text'].str.split().map(len).mean(), fine_chunks['text'].str.split().map(len).mean()],
})
display(chunk_summary.style.format({'avg_tokens': '{:.1f}'}))


### Visualize the Chunk Length Tradeoff

Fine chunks isolate evidence, while coarse chunks keep more context. Both behaviors show up in retrieval metrics.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(coarse_chunks['text'].str.split().map(len), bins=6, color='#72B7B2', edgecolor='white')
axes[0].set_title('Coarse chunk lengths')
axes[0].set_xlabel('Tokens')
axes[0].set_ylabel('Count')
axes[1].hist(fine_chunks['text'].str.split().map(len), bins=6, color='#E45756', edgecolor='white')
axes[1].set_title('Fine chunk lengths')
axes[1].set_xlabel('Tokens')
plt.tight_layout()
plt.show()


## 4. Tokenization Helpers

We will use a lightweight lowercase tokenizer. The goal is to understand retrieval behavior, not build a production tokenizer.


In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())


def build_vocabulary(texts):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = ['[PAD]', '[UNK]'] + sorted(counter)
    token_to_id = {token: idx for idx, token in enumerate(vocab)}
    return vocab, token_to_id

vocab, token_to_id = build_vocabulary(list(docs_df['text']) + list(queries_df['text']) + list(train_pairs_df['query_text']))
pad_idx = token_to_id['[PAD]']
unk_idx = token_to_id['[UNK]']
print(f'Vocabulary size: {len(vocab)}')


### Encode Text for the Dense Retriever

The dense model needs token ids and masks. Padding to a fixed length keeps the notebook simple.


In [ ]:
def encode_text(text, max_length):
    tokens = tokenize(text)[:max_length]
    token_ids = [token_to_id.get(token, unk_idx) for token in tokens]
    mask = [1] * len(token_ids)
    if len(token_ids) < max_length:
        padding = [pad_idx] * (max_length - len(token_ids))
        token_ids += padding
        mask += [0] * len(padding)
    return torch.tensor(token_ids, dtype=torch.long), torch.tensor(mask, dtype=torch.long)

sample_ids, sample_mask = encode_text(queries_df.iloc[0]['text'], CONFIG['max_length'])
print('Encoded shape:', sample_ids.shape, 'non-pad tokens:', int(sample_mask.sum()))


## 5. Sparse Retrieval Foundations

Before BM25, it helps to see the simplest lexical idea possible: count overlapping query terms.


In [ ]:
def overlap_score(query, document):
    return len(set(tokenize(query)) & set(tokenize(document)))

example_query = queries_df.iloc[2]['text']
example_view = docs_df[['doc_id', 'title']].copy()
example_view['overlap_score'] = docs_df['text'].map(lambda text: overlap_score(example_query, text))
example_view = example_view.sort_values('overlap_score', ascending=False)
print('Query:', example_query)
display(example_view.head(5))


### BM25 from Scratch

**BM25** improves lexical retrieval by weighting rare terms, saturating repeated matches, and normalizing for document length.


In [ ]:
class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(text) for text in texts]
        self.doc_lengths = np.array([len(doc) for doc in self.docs], dtype=np.float32)
        self.avg_doc_length = float(np.mean(self.doc_lengths))
        self.term_doc_freq = Counter()
        self.term_freqs = []
        for doc in self.docs:
            freq = Counter(doc)
            self.term_freqs.append(freq)
            for term in freq:
                self.term_doc_freq[term] += 1
        self.num_docs = len(self.docs)

    def idf(self, term):
        df = self.term_doc_freq.get(term, 0)
        return math.log(1 + (self.num_docs - df + 0.5) / (df + 0.5))

    def score(self, query, doc_index):
        score = 0.0
        doc_freqs = self.term_freqs[doc_index]
        doc_length = self.doc_lengths[doc_index]
        for term in tokenize(query):
            tf = doc_freqs.get(term, 0)
            if tf == 0:
                continue
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_length / self.avg_doc_length)
            score += self.idf(term) * numerator / denominator
        return score

    def rank(self, query, top_k=None):
        scores = np.array([self.score(query, idx) for idx in range(self.num_docs)])
        order = np.argsort(scores)[::-1]
        if top_k is not None:
            order = order[:top_k]
        return order, scores[order]


### Inspect BM25's Ingredients

The table and plots below show the three main BM25 intuitions in one place:

- rare words get higher **IDF**
- repeated matches help, but with diminishing returns
- longer documents are penalized when the same match is diluted across more text


In [ ]:
doc_bm25 = BM25(docs_df['text'].tolist())
terms_to_inspect = ['refund', 'reimbursements', 'package', 'authentication', 'support']
idf_table = pd.DataFrame({'term': terms_to_inspect, 'idf': [doc_bm25.idf(term) for term in terms_to_inspect]})
display(idf_table.style.format({'idf': '{:.3f}'}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
term_frequencies = np.arange(1, 11)
bm25_gain = term_frequencies * (1.5 + 1) / (term_frequencies + 1.5 * (1 - 0.75 + 0.75 * 1.0))
axes[0].plot(term_frequencies, bm25_gain, marker='o')
axes[0].set_title('BM25 tf saturation')
axes[0].set_xlabel('Term frequency in one document')
axes[0].set_ylabel('Contribution to score')

length_ratios = np.linspace(0.5, 2.0, 20)
axes[1].plot(length_ratios, 1 - 0.75 + 0.75 * length_ratios, color='#E45756')
axes[1].set_title('Length normalization term')
axes[1].set_xlabel('Document length / average length')
axes[1].set_ylabel('Normalization factor')
plt.tight_layout()
plt.show()


### Run BM25 and Show a Lexical Failure Mode

BM25 is strongest when the query uses the same words as the documents. It struggles on paraphrases like **credentials** vs **password** or **reimbursement** vs **refund**.


In [ ]:
def rank_doc_frame(index, frame, query, top_k=5):
    ranked_indices, ranked_scores = index.rank(query, top_k=top_k)
    ranked = frame.iloc[ranked_indices][['doc_id', 'title']].copy()
    ranked['score'] = ranked_scores
    return ranked

print('BM25 on semantic-synonym query')
display(rank_doc_frame(doc_bm25, docs_df, queries_df.loc[0, 'text']))
print('BM25 on credentials/password query')
display(rank_doc_frame(doc_bm25, docs_df, queries_df.loc[4, 'text']))


### Chunking Changes Lexical Retrieval

The same BM25 method behaves differently under different chunking strategies because the retriever now sees different search units.


In [ ]:
coarse_bm25 = BM25(coarse_chunks['text'].tolist())
fine_bm25 = BM25(fine_chunks['text'].tolist())


def chunk_ranked_doc_ids(index, chunk_frame, query, top_k):
    ranked_indices, _ = index.rank(query, top_k=top_k)
    return chunk_frame.iloc[ranked_indices]['doc_id'].tolist()


def recall_at_k_doc_level(relevant_docs, retrieved_doc_ids):
    return len(set(relevant_docs) & set(retrieved_doc_ids)) / len(set(relevant_docs))

chunking_rows = []
for _, row in queries_df.iterrows():
    coarse_docs = chunk_ranked_doc_ids(coarse_bm25, coarse_chunks, row['text'], CONFIG['top_k'])
    fine_docs = chunk_ranked_doc_ids(fine_bm25, fine_chunks, row['text'], CONFIG['top_k'])
    chunking_rows.append({
        'query_id': row['query_id'],
        'coarse_recall@3': recall_at_k_doc_level(row['relevant_docs'], coarse_docs),
        'fine_recall@3': recall_at_k_doc_level(row['relevant_docs'], fine_docs),
    })
chunking_results = pd.DataFrame(chunking_rows)
chunking_summary = pd.DataFrame({
    'strategy': ['Coarse chunks', 'Fine chunks'],
    'avg_recall@3': [chunking_results['coarse_recall@3'].mean(), chunking_results['fine_recall@3'].mean()],
})
display(chunking_results)
display(chunking_summary.style.format({'avg_recall@3': '{:.2%}'}))


## 6. Dense Retrieval Foundations

Dense retrieval represents text as vectors, so it can connect semantically related phrases even when exact keywords do not match.


In [ ]:
VAL_QUERY_IDS = ['q_security_alert', 'q_exchange_policy']

class RetrievalPairDataset(Dataset):
    def __init__(self, pair_frame, document_frame):
        lookup = document_frame.set_index('doc_id')
        self.samples = []
        for _, row in pair_frame.iterrows():
            doc_text = lookup.loc[row['doc_id'], 'text']
            self.samples.append((row['query_text'], doc_text))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        query_text, doc_text = self.samples[idx]
        query_ids, query_mask = encode_text(query_text, CONFIG['max_length'])
        doc_ids, doc_mask = encode_text(doc_text, CONFIG['max_length'])
        return query_ids, query_mask, doc_ids, doc_mask

train_dataset = RetrievalPairDataset(train_pairs_df, docs_df)
val_pairs = queries_df[queries_df['query_id'].isin(VAL_QUERY_IDS)].copy()
val_pairs = val_pairs.assign(query_text=val_pairs['text'], doc_id=val_pairs['relevant_docs'].map(lambda docs: docs[0]))
val_dataset = RetrievalPairDataset(val_pairs[['query_text', 'doc_id']], docs_df)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
print('Train pairs:', len(train_dataset), 'Validation pairs:', len(val_dataset))


### Implement a Tiny Dual Encoder

We will use a shared text encoder: token embeddings, masked mean pooling, and a projection layer. That is enough to demonstrate contrastive retrieval without turning this notebook into a model-architecture deep dive.


In [ ]:
class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, projection_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.projection = nn.Linear(embedding_dim, projection_dim)

    def forward(self, input_ids, attention_mask):
        embeddings = self.embedding(input_ids)
        mask = attention_mask.unsqueeze(-1)
        pooled = (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        projected = self.projection(pooled)
        return F.normalize(projected, dim=-1)


class DualEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, projection_dim):
        super().__init__()
        self.encoder = TinyTextEncoder(vocab_size, embedding_dim, projection_dim)

    def encode_queries(self, input_ids, attention_mask):
        return self.encoder(input_ids, attention_mask)

    def encode_docs(self, input_ids, attention_mask):
        return self.encoder(input_ids, attention_mask)


class DualEncoderModule(L.LightningModule):
    def __init__(self, model, learning_rate, temperature):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.temperature = temperature

    def forward(self, query_ids, query_mask, doc_ids, doc_mask):
        query_embeddings = self.model.encode_queries(query_ids, query_mask)
        doc_embeddings = self.model.encode_docs(doc_ids, doc_mask)
        return query_embeddings, doc_embeddings

    def _shared_step(self, batch, stage):
        query_ids, query_mask, doc_ids, doc_mask = batch
        query_embeddings, doc_embeddings = self(query_ids, query_mask, doc_ids, doc_mask)
        logits = query_embeddings @ doc_embeddings.T / self.temperature
        labels = torch.arange(logits.size(0), device=logits.device)
        loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))
        accuracy = (logits.argmax(dim=1) == labels).float().mean()
        self.log(f'{stage}_loss', loss, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        self.log(f'{stage}_acc', accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, 'val')

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)


### Train the Dense Retriever

The dense model only needs a small amount of local supervision here. The goal is to show how retrieval embeddings are learned, not to maximize benchmark performance.


In [ ]:
dual_encoder = DualEncoder(len(vocab), CONFIG['embedding_dim'], CONFIG['projection_dim'])
module = DualEncoderModule(dual_encoder, CONFIG['learning_rate'], CONFIG['temperature'])
logger = CSVLogger('logs', name='retrieval_fundamentals', version='dense_retriever')
early_stop = EarlyStopping(monitor='val_loss', patience=CONFIG['early_stop_patience'], mode='min')
trainer = L.Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='cpu',
    devices=1,
    logger=logger,
    callbacks=[early_stop],
    deterministic=True,
    enable_progress_bar=False,
    enable_checkpointing=False,
    num_sanity_val_steps=0,
)
trainer.fit(module, train_loader, val_loader)
print('Training log dir:', logger.log_dir)


### Plot the Dense-Retrieval Curves

The validation metric here is an **in-batch retrieval accuracy proxy**. It tells us whether the model is learning to align matching query-document pairs.


In [ ]:
dense_metrics = pd.read_csv(f'{logger.log_dir}/metrics.csv').groupby('epoch').max(numeric_only=True).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dense_metrics['epoch'], dense_metrics['train_loss'], marker='o', label='Train')
axes[0].plot(dense_metrics['epoch'], dense_metrics['val_loss'], marker='s', label='Validation')
axes[0].set_title('Dense retriever loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(dense_metrics['epoch'], dense_metrics['train_acc'], marker='o', label='Train')
axes[1].plot(dense_metrics['epoch'], dense_metrics['val_acc'], marker='s', label='Validation')
axes[1].set_title('In-batch retrieval accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()


### Encode the Corpus and Compare a Semantic Query

Once the dual encoder is trained, retrieval becomes nearest-neighbor search in embedding space. This is where semantic queries should benefit.


In [ ]:
def encode_frame_texts(model, frame, text_column):
    outputs = []
    model.eval()
    with torch.no_grad():
        for text in frame[text_column]:
            token_ids, attention_mask = encode_text(text, CONFIG['max_length'])
            embedding = model.encode_docs(token_ids.unsqueeze(0), attention_mask.unsqueeze(0))
            outputs.append(embedding.squeeze(0).cpu().numpy())
    return np.vstack(outputs)


def dense_rank(query_embedding, document_embeddings, top_k=None):
    scores = document_embeddings @ query_embedding
    order = np.argsort(scores)[::-1]
    if top_k is not None:
        order = order[:top_k]
    return order, scores[order]

doc_embeddings = encode_frame_texts(module.model, docs_df, 'text')
query_embeddings = encode_frame_texts(module.model, queries_df.assign(text=queries_df['text']), 'text')
query_id_to_embedding = {qid: emb for qid, emb in zip(queries_df['query_id'], query_embeddings)}

semantic_query = queries_df[queries_df['query_id'] == 'q_password_help'].iloc[0]
order, scores = dense_rank(query_id_to_embedding['q_password_help'], doc_embeddings, top_k=5)
semantic_view = docs_df.iloc[order][['doc_id', 'title']].copy()
semantic_view['score'] = scores
print('Dense retrieval on semantic query:', semantic_query['text'])
display(semantic_view)


### Compare Sparse and Dense Rankings Side by Side

This is the key conceptual handoff: BM25 is matching words, while dense retrieval is matching meaning.


In [ ]:
comparison_query = 'q_cancel_plan'
query_text = queries_df.loc[queries_df['query_id'] == comparison_query, 'text'].item()
bm25_view = rank_doc_frame(doc_bm25, docs_df, query_text, top_k=3).rename(columns={'score': 'bm25_score'})
dense_order, dense_scores = dense_rank(query_id_to_embedding[comparison_query], doc_embeddings, top_k=3)
dense_view = docs_df.iloc[dense_order][['doc_id', 'title']].copy()
dense_view['dense_score'] = dense_scores
print('Query:', query_text)
print('BM25:')
display(bm25_view)
print('Dense:')
display(dense_view)


## 7. Retrieval Metrics

Retrieval quality is not one number. Different metrics answer different operational questions.


In [ ]:
def precision_at_k(relevant_docs, retrieved_docs, k):
    return len(set(relevant_docs) & set(retrieved_docs[:k])) / k


def recall_at_k(relevant_docs, retrieved_docs, k):
    return len(set(relevant_docs) & set(retrieved_docs[:k])) / len(set(relevant_docs))


def reciprocal_rank(relevant_docs, retrieved_docs):
    relevant_docs = set(relevant_docs)
    for rank, doc_id in enumerate(retrieved_docs, start=1):
        if doc_id in relevant_docs:
            return 1.0 / rank
    return 0.0


def dcg_at_k(relevant_docs, retrieved_docs, k):
    relevant_docs = set(relevant_docs)
    return sum((1.0 if doc_id in relevant_docs else 0.0) / math.log2(rank + 1)
               for rank, doc_id in enumerate(retrieved_docs[:k], start=1))


def ndcg_at_k(relevant_docs, retrieved_docs, k):
    ideal_dcg = dcg_at_k(relevant_docs, list(relevant_docs), min(k, len(relevant_docs)))
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(relevant_docs, retrieved_docs, k) / ideal_dcg

print('Example metrics')
example_relevant = ['doc_a', 'doc_b']
example_retrieved = ['doc_x', 'doc_b', 'doc_a']
print('Precision@3:', precision_at_k(example_relevant, example_retrieved, 3))
print('Recall@3:', recall_at_k(example_relevant, example_retrieved, 3))
print('MRR:', reciprocal_rank(example_relevant, example_retrieved))
print('nDCG@3:', ndcg_at_k(example_relevant, example_retrieved, 3))


### Evaluate BM25 and Exact Dense Retrieval

These metrics show the main tradeoff: sparse methods usually win on exact keywords, while dense methods help on paraphrases and semantic similarity.


In [ ]:
def evaluate_ranked_lists(query_frame, ranking_fn, top_k):
    rows = []
    for _, row in query_frame.iterrows():
        ranked_doc_ids = ranking_fn(row['text'], row['query_id'])
        rows.append({
            'query_id': row['query_id'],
            'precision@k': precision_at_k(row['relevant_docs'], ranked_doc_ids, top_k),
            'recall@k': recall_at_k(row['relevant_docs'], ranked_doc_ids, top_k),
            'mrr': reciprocal_rank(row['relevant_docs'], ranked_doc_ids),
            'ndcg@k': ndcg_at_k(row['relevant_docs'], ranked_doc_ids, top_k),
            'top_docs': ranked_doc_ids[:top_k],
        })
    return pd.DataFrame(rows)


def bm25_ranking_fn(query_text, query_id):
    order, _ = doc_bm25.rank(query_text, top_k=None)
    return docs_df.iloc[order]['doc_id'].tolist()


def dense_ranking_fn(query_text, query_id):
    order, _ = dense_rank(query_id_to_embedding[query_id], doc_embeddings, top_k=None)
    return docs_df.iloc[order]['doc_id'].tolist()

bm25_eval = evaluate_ranked_lists(queries_df, bm25_ranking_fn, CONFIG['top_k'])
dense_eval = evaluate_ranked_lists(queries_df, dense_ranking_fn, CONFIG['top_k'])
summary_table = pd.DataFrame([
    {'method': 'BM25', 'precision@3': bm25_eval['precision@k'].mean(), 'recall@3': bm25_eval['recall@k'].mean(), 'MRR': bm25_eval['mrr'].mean(), 'nDCG@3': bm25_eval['ndcg@k'].mean()},
    {'method': 'Dense exact', 'precision@3': dense_eval['precision@k'].mean(), 'recall@3': dense_eval['recall@k'].mean(), 'MRR': dense_eval['mrr'].mean(), 'nDCG@3': dense_eval['ndcg@k'].mean()},
])
display(summary_table.style.format({'precision@3': '{:.2%}', 'recall@3': '{:.2%}', 'MRR': '{:.2f}', 'nDCG@3': '{:.2%}'}))

metric_long = summary_table.melt(id_vars='method', var_name='metric', value_name='value')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=metric_long, x='metric', y='value', hue='method', ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title('BM25 vs dense exact retrieval')
plt.tight_layout()
plt.show()


### Inspect Per-Query Wins and Losses

Averages matter, but per-query comparisons reveal whether dense retrieval is helping with the intended lexical-mismatch cases.


In [ ]:
per_query_compare = bm25_eval[['query_id', 'recall@k', 'mrr']].rename(columns={'recall@k': 'bm25_recall@3', 'mrr': 'bm25_mrr'})
per_query_compare = per_query_compare.merge(
    dense_eval[['query_id', 'recall@k', 'mrr']].rename(columns={'recall@k': 'dense_recall@3', 'mrr': 'dense_mrr'}),
    on='query_id'
).merge(queries_df[['query_id', 'intent']], on='query_id')
display(per_query_compare)


## 8. ANN Intuition with IVF-Style Clustering

Exact dense search compares the query against every document vector. ANN methods speed this up by searching only promising parts of the embedding space.


In [ ]:
clusterer = KMeans(n_clusters=CONFIG['ann_num_clusters'], random_state=CONFIG['seed'], n_init=10)
doc_cluster_ids = clusterer.fit_predict(doc_embeddings)
cluster_centers = clusterer.cluster_centers_
cluster_frame = docs_df[['doc_id', 'title']].copy()
cluster_frame['cluster'] = doc_cluster_ids
display(cluster_frame.sort_values('cluster'))


### Search Only the Best Clusters and Measure the Tradeoff

This is a lightweight IVF-style ANN approximation: first rank clusters, then rank only documents inside the nearest clusters.


In [ ]:
def ann_dense_ranking_fn(query_text, query_id):
    query_embedding = query_id_to_embedding[query_id]
    cluster_scores = cluster_centers @ query_embedding
    chosen_clusters = np.argsort(cluster_scores)[::-1][:CONFIG['ann_clusters_to_search']]
    candidate_mask = np.isin(doc_cluster_ids, chosen_clusters)
    candidate_embeddings = doc_embeddings[candidate_mask]
    candidate_doc_ids = docs_df.loc[candidate_mask, 'doc_id'].tolist()
    scores = candidate_embeddings @ query_embedding
    order = np.argsort(scores)[::-1]
    ranked_doc_ids = [candidate_doc_ids[idx] for idx in order]
    remainder = [doc_id for doc_id in docs_df['doc_id'] if doc_id not in ranked_doc_ids]
    return ranked_doc_ids + remainder

ann_eval = evaluate_ranked_lists(queries_df, ann_dense_ranking_fn, CONFIG['top_k'])

def measure_latency(ranking_fn, query_frame, repeats):
    start = time.perf_counter()
    for _ in range(repeats):
        for _, row in query_frame.iterrows():
            ranking_fn(row['text'], row['query_id'])
    elapsed = time.perf_counter() - start
    return elapsed / (repeats * len(query_frame))

latency_table = pd.DataFrame([
    {'method': 'Dense exact', 'avg_query_latency_ms': measure_latency(dense_ranking_fn, queries_df, CONFIG['latency_repeats']) * 1000, 'recall@3': dense_eval['recall@k'].mean()},
    {'method': 'Dense ANN', 'avg_query_latency_ms': measure_latency(ann_dense_ranking_fn, queries_df, CONFIG['latency_repeats']) * 1000, 'recall@3': ann_eval['recall@k'].mean()},
])
display(latency_table.style.format({'avg_query_latency_ms': '{:.4f}', 'recall@3': '{:.2%}'}))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(latency_table['avg_query_latency_ms'], latency_table['recall@3'], s=140, c=['#4C78A8', '#F58518'])
for _, row in latency_table.iterrows():
    ax.annotate(row['method'], (row['avg_query_latency_ms'], row['recall@3']), xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Average latency per query (ms)')
ax.set_ylabel('Average recall@3')
ax.set_title('ANN trades some recall for lower search cost')
plt.tight_layout()
plt.show()


## 9. Failure Analysis and Decision Matrix

Aggregate metrics are useful, but retrieval debugging usually requires reading the misses directly.


In [ ]:
def top_docs_lookup(eval_frame):
    return {row['query_id']: row['top_docs'] for _, row in eval_frame.iterrows()}

bm25_top_docs = top_docs_lookup(bm25_eval)
dense_top_docs = top_docs_lookup(dense_eval)
ann_top_docs = top_docs_lookup(ann_eval)

failure_rows = []
for _, row in queries_df.iterrows():
    relevant = set(row['relevant_docs'])
    failure_rows.append({
        'query_id': row['query_id'],
        'intent': row['intent'],
        'bm25_hit@1': bm25_top_docs[row['query_id']][0] in relevant,
        'dense_hit@1': dense_top_docs[row['query_id']][0] in relevant,
        'ann_hit@1': ann_top_docs[row['query_id']][0] in relevant,
        'bm25_top1': bm25_top_docs[row['query_id']][0],
        'dense_top1': dense_top_docs[row['query_id']][0],
        'ann_top1': ann_top_docs[row['query_id']][0],
    })

failure_table = pd.DataFrame(failure_rows)
display(failure_table)

decision_matrix = pd.DataFrame([
    ['BM25', 'Exact keywords, interpretable scores, zero training', 'Misses paraphrases and semantic similarity'],
    ['Dense retrieval', 'Handles synonyms and semantic overlap', 'Needs training data and embedding infrastructure'],
    ['Dense ANN', 'Lower latency at scale', 'Can lose recall if search is too approximate'],
    ['Fine chunks', 'Precise evidence retrieval', 'Context can become fragmented'],
    ['Coarse chunks', 'More context per hit', 'More distracting terms'],
], columns=['Choice', 'When it shines', 'Main downside'])
display(decision_matrix)


### Key Takeaways

This notebook taught the retrieval half of the problem. The next notebook, **RAG**, will reuse these ideas and add prompt construction, retrieved context formatting, and generation.


In [ ]:
print('Key takeaways:')
print('1. Retrieval quality starts with the unit of search: document vs chunk.')
print('2. BM25 is the core sparse baseline because it balances exact match, rarity, and document length.')
print('3. Dense retrieval learns semantic similarity from labeled query-document pairs.')
print('4. ANN indexes reduce search cost by scanning only promising embedding regions.')
print('5. Recall@k, Precision@k, MRR, and nDCG answer different evaluation questions.')
print('6. Hybrid retrieval and retrieve-then-generate pipelines belong in the RAG notebook, not here.')
